# STCP - Queries Operacionais

Este notebook responde a perguntas concretas sobre o estado actual da frota em tempo real, com base nos dados que chegam do simulador Python via Azure IoT Central e Azure Event Hub para as tabelas Delta `bus_telemetry` e `charger_telemetry`.

## Estrutura
- Grupo 1: Autocarros
- Grupo 2: Estações de Carregamento
- Grupo 3: Serviços
- Grupo 4: Queries Combinadas

# Grupo 1 - Autocarros

## Estado Actual de Todos os Autocarros

In [0]:
# Devolve o registo mais recente de cada autocarro
# ROW_NUMBER() garante que apenas aparece uma linha por autocarro, a mais recente

display(spark.sql("""
    SELECT
        device_id,
        state,
        round(batteryLevel, 2) as bateria_percentagem,
        round(batteryKwh, 2) as bateria_kwh,
        round(batteryTemperature, 1) as temperatura_celsius,
        round(latitude, 6) as latitude,
        round(longitude, 6) as longitude,
        date_format(event_time, 'dd-MM-yyyy HH:mm:ss') as ultima_actualizacao
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry_raw
    )
    WHERE rn = 1
    ORDER BY device_id
"""))

device_id,state,bateria_percentagem,bateria_kwh,temperatura_celsius,latitude,longitude,ultima_actualizacao
BUS-001,inTransit,47.29,165.51,23.7,41.157517,-8.678186,30-03-2026 17:00:04
BUS-002,inTransit,71.69,250.91,22.3,41.150297,-8.613119,30-03-2026 17:00:04
BUS-003,inTransit,25.49,89.21,23.8,41.151786,-8.673122,30-03-2026 17:00:04


## Autocarro com a Bateria Mais Baixa

In [0]:
# Identifica o autocarro mais urgente em termos de necessidade de carregamento
# Inclui alerta com classificacao: CRITICO abaixo de 20%, BAIXO abaixo de 50%, OK acima de 50%

display(spark.sql("""
    SELECT
        device_id,
        state,
        round(batteryLevel, 2) as bateria_percentagem,
        round(batteryKwh, 2) as bateria_kwh,
        CASE
            WHEN batteryLevel < 20 THEN 'CRITICO - abaixo do minimo de seguranca'
            WHEN batteryLevel < 50 THEN 'BAIXO - necessita carregamento'
            ELSE 'OK'
        END as alerta
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry_raw
    )
    WHERE rn = 1
    ORDER BY batteryLevel ASC
    LIMIT 1
"""))

device_id,state,bateria_percentagem,bateria_kwh,alerta
BUS-003,inTransit,25.49,89.21,BAIXO - necessita carregamento


## Autocarros em Nível Crítico (abaixo de 20%)

In [0]:
# Lista todos os autocarros com bateria abaixo do nivel minimo de seguranca (20%)
# Estes autocarros nao devem sair em rota sem carregar primeiro

display(spark.sql("""
    SELECT
        device_id,
        state,
        round(batteryLevel, 2) as bateria_percentagem,
        round(batteryKwh, 2) as bateria_kwh,
        round(latitude, 6) as latitude,
        round(longitude, 6) as longitude,
        date_format(event_time, 'dd-MM-yyyy HH:mm:ss') as ultima_actualizacao
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry_raw
    )
    WHERE rn = 1
    AND batteryLevel < 20
    ORDER BY batteryLevel ASC
"""))

device_id,state,bateria_percentagem,bateria_kwh,latitude,longitude,ultima_actualizacao


## Autonomia Restante de Cada Autocarro

In [0]:
# Autonomia restante = bateria disponivel (kWh) / consumo medio (kWh/km)
# Consumo medio: 1.2 kWh/km (valor tipico urbano para CaetanoBus e.City Gold)
# Exemplo: BUS-003 com 87.5 kWh -> 87.5 / 1.2 = 72.9 km restantes

display(spark.sql("""
    SELECT
        device_id,
        state,
        round(batteryLevel, 2) as bateria_percentagem,
        round(batteryKwh, 2) as bateria_kwh,
        round(batteryKwh / 1.2, 2) as autonomia_restante_km,
        CASE
            WHEN batteryKwh / 1.2 < 20 THEN 'CRITICO - autonomia insuficiente'
            WHEN batteryKwh / 1.2 < 50 THEN 'BAIXO - autonomia reduzida'
            ELSE 'OK'
        END as alerta
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry_raw
    )
    WHERE rn = 1
    ORDER BY batteryKwh ASC
"""))

device_id,state,bateria_percentagem,bateria_kwh,autonomia_restante_km,alerta
BUS-003,inTransit,25.49,89.21,74.34,OK
BUS-001,inTransit,47.29,165.51,137.93,OK
BUS-002,inTransit,71.69,250.91,209.09,OK


## Vista Completa de Cada Autocarro

In [0]:
# Vista consolidada com todos os campos relevantes de cada autocarro
# Autonomia calculada com consumo medio de 1.2 kWh/km
# Estado da bateria: CRITICO abaixo de 20%, BAIXO abaixo de 50%, OK acima de 50%

display(spark.sql("""
    SELECT
        device_id,
        state,
        round(batteryLevel, 2) as bateria_percentagem,
        round(batteryKwh, 2) as bateria_kwh,
        round(batteryKwh / 1.2, 2) as autonomia_km,
        round(batteryTemperature, 1) as temperatura_celsius,
        round(latitude, 6) as latitude,
        round(longitude, 6) as longitude,
        CASE
            WHEN batteryLevel < 20 THEN 'CRITICO'
            WHEN batteryLevel < 50 THEN 'BAIXO'
            ELSE 'OK'
        END as estado_bateria,
        date_format(event_time, 'dd-MM-yyyy HH:mm:ss') as ultima_actualizacao
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry_raw
    )
    WHERE rn = 1
    ORDER BY device_id
"""))

device_id,state,bateria_percentagem,bateria_kwh,autonomia_km,temperatura_celsius,latitude,longitude,estado_bateria,ultima_actualizacao
BUS-001,inTransit,47.29,165.5,137.92,24.2,41.157461,-8.676845,BAIXO,30-03-2026 17:01:16
BUS-002,inTransit,71.69,250.9,209.08,22.8,41.148819,-8.614507,OK,30-03-2026 17:01:16
BUS-003,inTransit,25.49,89.2,74.33,24.3,41.151477,-8.673199,BAIXO,30-03-2026 17:01:12


## Evolução da Bateria do BUS-003 (últimas 10 leituras)

In [0]:
# Mostra a evolucao da bateria do BUS-003 ao longo do tempo
# Permite verificar que a bateria esta efectivamente a descer enquanto o autocarro esta em rota
# Autonomia calculada com consumo medio de 1.2 kWh/km

display(spark.sql("""
    SELECT
        device_id,
        round(batteryLevel, 2) as bateria_percentagem,
        round(batteryKwh, 2) as bateria_kwh,
        round(batteryKwh / 1.2, 2) as autonomia_km,
        state,
        date_format(event_time, 'dd-MM-yyyy HH:mm:ss') as hora
    FROM bus_telemetry_raw
    WHERE device_id = 'BUS-003'
    ORDER BY event_time DESC
    LIMIT 10
"""))

device_id,bateria_percentagem,bateria_kwh,autonomia_km,state,hora
BUS-003,25.49,89.2,74.33,inTransit,30-03-2026 17:01:51
BUS-003,25.49,89.2,74.33,inTransit,30-03-2026 17:01:48
BUS-003,25.49,89.2,74.33,inTransit,30-03-2026 17:01:41
BUS-003,25.49,89.2,74.33,inTransit,30-03-2026 17:01:37
BUS-003,25.49,89.2,74.33,inTransit,30-03-2026 17:01:32
BUS-003,25.49,89.2,74.33,inTransit,30-03-2026 17:01:26
BUS-003,25.49,89.2,74.33,inTransit,30-03-2026 17:01:21
BUS-003,25.49,89.2,74.33,inTransit,30-03-2026 17:01:19
BUS-003,25.49,89.2,74.33,inTransit,30-03-2026 17:01:12
BUS-003,25.49,89.21,74.34,inTransit,30-03-2026 17:01:05


# Grupo 2 - Estações de Carregamento

## Estado Actual de Todas as Estações

In [0]:
# Devolve o estado mais recente de cada estacao de carregamento
# Estados possiveis: available (livre), occupied (a carregar), fault (avaria)

display(spark.sql("""
    SELECT
        device_id,
        state,
        round(currentPower, 1) as potencia_actual_kw,
        round(currentEfficiency, 1) as eficiencia_percentagem,
        round(chargerTemperature, 1) as temperatura_celsius,
        round(energyDelivered, 2) as energia_entregue_kwh,
        connectedBus as autocarro_ligado,
        date_format(event_time, 'dd-MM-yyyy HH:mm:ss') as ultima_actualizacao
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry_raw
    )
    WHERE rn = 1
    ORDER BY device_id
"""))

device_id,state,potencia_actual_kw,eficiencia_percentagem,temperatura_celsius,energia_entregue_kwh,autocarro_ligado,ultima_actualizacao
CS-001,occupied,80.0,91.6,42.8,21.5,BUS-004,30-03-2026 17:02:16
CS-002,fault,0.0,0.0,28.2,0.0,null,30-03-2026 17:02:18
CS-003,available,0.0,92.0,22.0,0.0,null,30-03-2026 17:02:16


## Estações Disponíveis para Carregamento

In [0]:
# Lista apenas as estacoes com estado available, prontas para receber um autocarro
# Estacoes em fault ou occupied nao aparecem nesta lista

display(spark.sql("""
    SELECT
        device_id,
        state,
        round(chargerTemperature, 1) as temperatura_celsius,
        round(currentEfficiency, 1) as eficiencia_percentagem
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry_raw
    )
    WHERE rn = 1
    AND state = 'available'
    ORDER BY device_id
"""))

device_id,state,temperatura_celsius,eficiencia_percentagem
CS-003,available,22.0,92.0


## Estações com Avaria

In [0]:
# Lista as estacoes com estado fault, ou seja, com avaria e fora de servico
# Estas estacoes nao podem receber autocarros ate serem reparadas

display(spark.sql("""
    SELECT
        device_id,
        state,
        round(chargerTemperature, 1) as temperatura_celsius,
        date_format(event_time, 'dd-MM-yyyy HH:mm:ss') as ultima_actualizacao
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry_raw
    )
    WHERE rn = 1
    AND state = 'fault'
    ORDER BY device_id
"""))

device_id,state,temperatura_celsius,ultima_actualizacao
CS-002,fault,28.2,30-03-2026 17:02:38


# Grupo 3 - Serviços

## Capacidade Disponível na Rede de Carregamento

In [0]:
# Calcula a capacidade total da rede de carregamento e quanto ainda está disponível
# Potência máxima por estação: 120 kW
# Potência total da rede: 300 kW (3 estações x 120 kW)
# Estações em fault não contribuem para a capacidade disponível

display(spark.sql("""
    SELECT
        COUNT(*) as total_estacoes,
        SUM(CASE WHEN state = 'available' THEN 1 ELSE 0 END) as estacoes_livres,
        SUM(CASE WHEN state = 'occupied' THEN 1 ELSE 0 END) as estacoes_ocupadas,
        SUM(CASE WHEN state = 'fault' THEN 1 ELSE 0 END) as estacoes_avaria,
        round(SUM(CASE WHEN state != 'fault' THEN 120 ELSE 0 END), 1) as capacidade_total_kw,
        round(SUM(CASE WHEN state = 'occupied' THEN currentPower ELSE 0 END), 1) as potencia_em_uso_kw,
        round(SUM(CASE WHEN state = 'available' THEN 120 ELSE 0 END), 1) as capacidade_disponivel_kw
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry_raw
    )
    WHERE rn = 1
"""))

total_estacoes,estacoes_livres,estacoes_ocupadas,estacoes_avaria,capacidade_total_kw,potencia_em_uso_kw,capacidade_disponivel_kw
3,1,1,1,240,80.0,120


## Tempo Estimado para Carregar Cada Autocarro até 80%

In [0]:
# Calcula o tempo estimado para carregar cada autocarro até ao nível ideal de 80%
# Fórmula: tempo (horas) = energia necessária (kWh) / potência de carregamento (kW)
# Potência de carregamento assumida: 80 kW (valor usado pelo orquestrador)
# Eficiência de carregamento: 92%
# Apenas calculado para autocarros com bateria abaixo de 80%

display(spark.sql("""
    SELECT
        device_id,
        state,
        round(batteryLevel, 2) as bateria_actual_percentagem,
        round(batteryKwh, 2) as bateria_actual_kwh,
        round(350 * 0.80, 1) as bateria_objetivo_kwh,
        round((350 * 0.80) - batteryKwh, 2) as energia_necessaria_kwh,
        CASE
            WHEN batteryLevel >= 80 THEN 'Ja carregado'
            ELSE concat(
                cast(round(((350 * 0.80) - batteryKwh) / (80 * 0.92), 2) as string),
                ' horas'
            )
        END as tempo_estimado_carregamento
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry_raw
    )
    WHERE rn = 1
    ORDER BY batteryLevel ASC
"""))

device_id,state,bateria_actual_percentagem,bateria_actual_kwh,bateria_objetivo_kwh,energia_necessaria_kwh,tempo_estimado_carregamento
BUS-003,inTransit,25.5,89.25,280.0,190.75,2.59 horas
BUS-001,inTransit,47.3,165.55,280.0,114.45,1.56 horas
BUS-002,inTransit,71.7,250.95,280.0,29.05,0.39 horas


## Autocarros que Precisam de Carregar antes da Próxima Rota

In [0]:
# Identifica autocarros cuja bateria actual é insuficiente para completar a rota atribuída
# Energia necessária por rota (baseada nos dados do Excel do projecto):
#   Linha 200 (BUS-001): 45 km x 1.2 kWh/km = 54 kWh (+ 20% margem de segurança = 64.8 kWh)
#   Linha 201 (BUS-002): 38 km x 1.2 kWh/km = 45.6 kWh (+ 20% margem de segurança = 54.72 kWh)
#   Linha 202 (BUS-003): 52 km x 1.2 kWh/km = 62.4 kWh (+ 20% margem de segurança = 74.88 kWh)

display(spark.sql("""
    SELECT
        device_id,
        state,
        round(batteryLevel, 2) as bateria_actual_percentagem,
        round(batteryKwh, 2) as bateria_actual_kwh,
        CASE device_id
            WHEN 'BUS-001' THEN 64.8
            WHEN 'BUS-002' THEN 54.72
            WHEN 'BUS-003' THEN 74.88
        END as energia_necessaria_kwh,
        CASE
            WHEN batteryKwh >= CASE device_id
                WHEN 'BUS-001' THEN 64.8
                WHEN 'BUS-002' THEN 54.72
                WHEN 'BUS-003' THEN 74.88
            END THEN 'OK - bateria suficiente'
            ELSE 'NECESSITA CARREGAR antes da rota'
        END as estado_operacional
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry_raw
    )
    WHERE rn = 1
    ORDER BY device_id
"""))

device_id,state,bateria_actual_percentagem,bateria_actual_kwh,energia_necessaria_kwh,estado_operacional
BUS-001,inTransit,47.3,165.55,64.80,OK - bateria suficiente
BUS-002,inTransit,71.7,250.95,54.72,OK - bateria suficiente
BUS-003,inTransit,25.5,89.25,74.88,OK - bateria suficiente


## Paragem Mais Próxima de Cada Autocarro

In [0]:
# Identifica a paragem actual de cada autocarro com base na distancia percorrida
# em vez do GPS, que é aproximado no simulador.
# A paragem actual é a que tem distance_meters mais próximo do distanceTraveled actual.
# Usa a tabela route_stops criada a partir dos dados GTFS reais das linhas 200, 201 e 202.

display(spark.sql("""
    WITH posicoes_actuais AS (
        SELECT device_id, latitude, longitude, batteryLevel, state, distanceTraveled
        FROM (
            SELECT *,
                ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
            FROM bus_telemetry_raw
        )
        WHERE rn = 1
    ),
    paragem_proxima AS (
        SELECT
            b.device_id,
            b.state as estado_autocarro,
            round(b.batteryLevel, 2) as bateria_percentagem,
            round(b.latitude, 6) as autocarro_lat,
            round(b.longitude, 6) as autocarro_lon,
            round(b.distanceTraveled, 0) as distancia_percorrida_metros,
            p.stop_sequence as numero_paragem,
            p.stop_name as paragem_mais_proxima,
            p.arrival_time as hora_prevista,
            round(p.distance_meters, 0) as distancia_paragem_metros,
            round(p.stop_lat, 6) as paragem_lat,
            round(p.stop_lon, 6) as paragem_lon,
            ROW_NUMBER() OVER (
                PARTITION BY b.device_id
                ORDER BY ABS(b.distanceTraveled - p.distance_meters) ASC
            ) as rank
        FROM posicoes_actuais b
        JOIN route_stops p ON b.device_id = p.bus_id
    )
    SELECT
        device_id as autocarro,
        estado_autocarro,
        bateria_percentagem,
        autocarro_lat,
        autocarro_lon,
        distancia_percorrida_metros,
        numero_paragem,
        paragem_mais_proxima,
        hora_prevista,
        distancia_paragem_metros,
        paragem_lat,
        paragem_lon
    FROM paragem_proxima
    WHERE rank = 1
    ORDER BY device_id
"""))

autocarro,estado_autocarro,bateria_percentagem,autocarro_lat,autocarro_lon,distancia_percorrida_metros,numero_paragem,paragem_mais_proxima,hora_prevista,distancia_paragem_metros,paragem_lat,paragem_lon
BUS-001,inTransit,47.28,41.157486,-8.676964,212099.0,30,Castelo Do Queijo,06:27,9854.0,41.167422,-8.689127
BUS-002,inTransit,71.68,41.147031,-8.61388,204697.0,26,Viso,08:18,9778.0,41.17784,-8.64228
BUS-003,inTransit,25.48,41.153497,-8.673813,209313.0,16,Passeio Alegre,06:57,5313.0,41.14882,-8.67275


## Quantas Paragens Faltam para Cada Autocarro Terminar a Rota

In [0]:
# Calcula quantas paragens faltam para cada autocarro terminar a rota
# baseado na distancia percorrida actual e nas distancias das paragens reais do GTFS
# Total de paragens: BUS-001 tem 30 (Linha 200), BUS-002 tem 26 (Linha 201), BUS-003 tem 16 (Linha 202)

display(spark.sql("""
    WITH posicoes_actuais AS (
        SELECT device_id, batteryLevel, state, distanceTraveled
        FROM (
            SELECT *,
                ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
            FROM bus_telemetry_raw
        )
        WHERE rn = 1
    ),
    paragem_actual AS (
        SELECT
            b.device_id,
            b.state as estado_autocarro,
            round(b.batteryLevel, 2) as bateria_percentagem,
            round(b.distanceTraveled, 0) as distancia_percorrida_metros,
            p.stop_sequence as paragem_actual,
            p.stop_name as nome_paragem_actual,
            p.arrival_time as hora_prevista,
            ROW_NUMBER() OVER (
                PARTITION BY b.device_id
                ORDER BY ABS(b.distanceTraveled - p.distance_meters) ASC
            ) as rank
        FROM posicoes_actuais b
        JOIN route_stops p ON b.device_id = p.bus_id
    ),
    ultima_paragem AS (
        SELECT bus_id, MAX(stop_sequence) as total_paragens,
               FIRST_VALUE(stop_name) OVER (
                   PARTITION BY bus_id ORDER BY stop_sequence DESC
               ) as paragem_final,
               FIRST_VALUE(arrival_time) OVER (
                   PARTITION BY bus_id ORDER BY stop_sequence DESC
               ) as hora_chegada_prevista
        FROM route_stops
        GROUP BY bus_id, stop_name, arrival_time, stop_sequence
        QUALIFY ROW_NUMBER() OVER (PARTITION BY bus_id ORDER BY stop_sequence DESC) = 1
    )
    SELECT
        p.device_id as autocarro,
        p.estado_autocarro,
        p.bateria_percentagem,
        p.distancia_percorrida_metros,
        p.paragem_actual,
        p.nome_paragem_actual as paragem_mais_proxima,
        p.hora_prevista,
        u.total_paragens,
        (u.total_paragens - p.paragem_actual) as paragens_restantes,
        u.paragem_final as destino_final,
        u.hora_chegada_prevista as hora_chegada_prevista
    FROM paragem_actual p
    JOIN ultima_paragem u ON p.device_id = u.bus_id
    WHERE p.rank = 1
    ORDER BY p.device_id
"""))

autocarro,estado_autocarro,bateria_percentagem,distancia_percorrida_metros,paragem_actual,paragem_mais_proxima,hora_prevista,total_paragens,paragens_restantes,destino_final,hora_chegada_prevista
BUS-001,inTransit,47.28,216099.0,30,Castelo Do Queijo,06:27,30,0,Castelo Do Queijo,06:27
BUS-002,inTransit,71.68,208697.0,26,Viso,08:18,26,0,Viso,08:18
BUS-003,inTransit,25.48,213313.0,16,Passeio Alegre,06:57,16,0,Passeio Alegre,06:57


# Grupo 4 - Queries Combinadas

## Qual o Autocarro que Está a Ser Carregado em Cada Estação

In [0]:
# Cruza os dados das estações com os dados dos autocarros
# para mostrar o estado completo de cada sessão de carregamento activa
# Inclui informação do autocarro (bateria) e da estação (potência, energia entregue)

display(spark.sql("""
    WITH estacoes_ocupadas AS (
        SELECT device_id as estacao_id, connectedBus, currentPower, energyDelivered, chargerTemperature
        FROM (
            SELECT *,
                ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
            FROM charger_telemetry_raw
        )
        WHERE rn = 1
        AND state = 'occupied'
    ),
    autocarros AS (
        SELECT device_id, batteryLevel, batteryKwh, state as estado_autocarro
        FROM (
            SELECT *,
                ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
            FROM bus_telemetry_raw
        )
        WHERE rn = 1
    )
    SELECT
        e.estacao_id as estacao,
        e.connectedBus as autocarro,
        e.currentPower as potencia_kw,
        round(e.energyDelivered, 2) as energia_entregue_kwh,
        round(e.chargerTemperature, 1) as temperatura_estacao_celsius,
        round(a.batteryLevel, 2) as bateria_autocarro_percentagem,
        round(a.batteryKwh, 2) as bateria_autocarro_kwh,
        a.estado_autocarro
    FROM estacoes_ocupadas e
    LEFT JOIN autocarros a ON e.connectedBus = a.device_id
    ORDER BY e.estacao_id
"""))

estacao,autocarro,potencia_kw,energia_entregue_kwh,temperatura_estacao_celsius,bateria_autocarro_percentagem,bateria_autocarro_kwh,estado_autocarro
CS-001,BUS-004,80.0,15.06,37.5,null,null,null


## Há Estações Suficientes para os Autocarros que Precisam de Carregar?

In [0]:
# Verifica se existe capacidade de carregamento disponível para todos os autocarros
# que precisam de carregar (bateria abaixo de 50%)
# Cruza o número de autocarros que precisam de carregar com o número de estações disponíveis

display(spark.sql("""
    WITH autocarros_precisam AS (
        SELECT COUNT(*) as autocarros_precisam_carregar
        FROM (
            SELECT *,
                ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
            FROM bus_telemetry_raw
        )
        WHERE rn = 1
        AND batteryLevel < 50
    ),
    estacoes_livres AS (
        SELECT COUNT(*) as estacoes_disponiveis
        FROM (
            SELECT *,
                ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
            FROM charger_telemetry_raw
        )
        WHERE rn = 1
        AND state = 'available'
    )
    SELECT
        a.autocarros_precisam_carregar,
        e.estacoes_disponiveis,
        CASE
            WHEN e.estacoes_disponiveis >= a.autocarros_precisam_carregar THEN 'OK - capacidade suficiente'
            WHEN e.estacoes_disponiveis > 0 THEN 'ATENCAO - capacidade parcial'
            ELSE 'CRITICO - sem estacoes disponiveis'
        END as estado_capacidade,
        (e.estacoes_disponiveis - a.autocarros_precisam_carregar) as estacoes_excedentes
    FROM autocarros_precisam a, estacoes_livres e
"""))

autocarros_precisam_carregar,estacoes_disponiveis,estado_capacidade,estacoes_excedentes
2,1,ATENCAO - capacidade parcial,-1


## Custo Estimado para Carregar Todos os Autocarros até 80%

In [0]:
# Calcula o custo estimado de energia para carregar todos os autocarros até ao nível ideal de 80%
# Tarifa base: 0.15 euros por kWh (definida no simulador)
# Eficiência de carregamento: 92% (definida no simulador)
# Apenas considera autocarros com bateria abaixo de 80%

display(spark.sql("""
    SELECT
        device_id as autocarro,
        round(batteryLevel, 2) as bateria_actual_percentagem,
        round(batteryKwh, 2) as bateria_actual_kwh,
        round(350 * 0.80, 1) as bateria_objetivo_kwh,
        round((350 * 0.80) - batteryKwh, 2) as energia_necessaria_kwh,
        round(((350 * 0.80) - batteryKwh) / 0.92, 2) as energia_rede_necessaria_kwh,
        round((((350 * 0.80) - batteryKwh) / 0.92) * 0.15, 2) as custo_estimado_euros,
        round(((350 * 0.80) - batteryKwh) / (80 * 0.92), 2) as tempo_estimado_horas
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry_raw
    )
    WHERE rn = 1
    AND batteryLevel < 80
    ORDER BY batteryLevel ASC
"""))

autocarro,bateria_actual_percentagem,bateria_actual_kwh,bateria_objetivo_kwh,energia_necessaria_kwh,energia_rede_necessaria_kwh,custo_estimado_euros,tempo_estimado_horas
BUS-003,25.5,89.25,280.0,190.75,207.34,31.1,2.59
BUS-001,47.3,165.55,280.0,114.45,124.4,18.66,1.56
BUS-002,71.7,250.95,280.0,29.05,31.58,4.74,0.39
